## Imports and Declarations

In [1]:
import pandas as pd
import numpy as np
import torch
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer

/opt/anaconda3/envs/lumaa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#search_sentence = "I love thrilling action movies set in space, with a comedic twist."
#search_sentence = "I love movies based on gangster, cartels and crime that have drama and action in it with great storyline"
search_sentence = "I love coming of age movies with a comedic twist."

In [3]:
tfidf = TfidfVectorizer()
bert_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
model = AutoModel.from_pretrained(bert_model_name)

In [4]:
nltk.download('wordnet')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ritamupadhyay/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
top_k_matches = 5

## Data Import

### Data Set is available at https://www.kaggle.com/datasets/adikhare/top-100-imdb-movies

In [6]:
df = pd.read_csv('data/Top 100 IMDB Movies.csv')

In [7]:
df

,rank,title,description,image,big_image,genre,thumbnail,rating,id,year,imdbid,imdb_link
0,1,The Shawshank Redemption,Two imprisoned men bond over a number of years...,https://m.media-amazon.com/images/M/MV5BMDFkYT...,https://m.media-amazon.com/images/M/MV5BMDFkYT...,['Drama'],https://m.media-amazon.com/images/M/MV5BMDFkYT...,9.3,top1,1994,tt0111161,https://www.imdb.com/title/tt0111161
1,2,The Godfather,The aging patriarch of an organized crime dyna...,https://m.media-amazon.com/images/M/MV5BM2MyNj...,https://m.media-amazon.com/images/M/MV5BM2MyNj...,"['Crime', 'Drama']",https://m.media-amazon.com/images/M/MV5BM2MyNj...,9.2,top2,1972,tt0068646,https://www.imdb.com/title/tt0068646
2,3,The Dark Knight,When the menace known as the Joker wreaks havo...,https://m.media-amazon.com/images/M/MV5BMTMxNT...,https://m.media-amazon.com/images/M/MV5BMTMxNT...,"['Action', 'Crime', 'Drama']",https://m.media-amazon.com/images/M/MV5BMTMxNT...,9.0,top3,2008,tt0468569,https://www.imdb.com/title/tt0468569
3,4,The Godfather Part II,The early life and career of Vito Corleone in ...,https://m.media-amazon.com/images/M/MV5BMWMwMG...,https://m.media-amazon.com/images/M/MV5BMWMwMG...,"['Crime', 'Drama']",https://m.media-amazon.com/images/M/MV5BMWMwMG...,9.0,top4,1974,tt0071562,https://www.imdb.com/title/tt0071562
4,5,12 Angry Men,The jury in a New York City murder trial is fr...,https://m.media-amazon.com/images/M/MV5BMWU4N2...,https://m.media-amazon.com/images/M/MV5BMWU4N2...,"['Crime', 'Drama']",https://m.media-amazon.com/images/M/MV5BMWU4N2...,9.0,top5,1957,tt0050083,https://www.imdb.com/title/tt0050083
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,Reservoir Dogs,When a simple jewelry heist goes horribly wron...,https://m.media-amazon.com/images/M/MV5BZmExNm...,https://m.media-amazon.com/images/M/MV5BZmExNm...,"['Crime', 'Thriller']",https://m.media-amazon.com/images/M/MV5BZmExNm...,8.3,top96,1992,tt0105236,https://www.imdb.com/title/tt0105236
96,97,Ikiru,A bureaucrat tries to find meaning in his life...,https://m.media-amazon.com/images/M/MV5BYWM1Ym...,https://m.media-amazon.com/images/M/MV5BYWM1Ym...,['Drama'],https://m.media-amazon.com/images/M/MV5BYWM1Ym...,8.3,top97,1952,tt0044741,https://www.imdb.com/title/tt0044741
97,98,Lawrence of Arabia,"The story of T.E. Lawrence, the English office...",https://m.media-amazon.com/images/M/MV5BYWY5Zj...,https://m.media-amazon.com/images/M/MV5BYWY5Zj...,"['Adventure', 'Biography', 'Drama']",https://m.media-amazon.com/images/M/MV5BYWY5Zj...,8.3,top98,1962,tt0056172,https://www.imdb.com/title/tt0056172
98,99,Citizen Kane,Following the death of publishing tycoon Charl...,https://m.media-amazon.com/images/M/MV5BYjBiOT...,https://m.media-amazon.com/images/M/MV5BYjBiOT...,"['Drama', 'Mystery']",https://m.media-amazon.com/images/M/MV5BYjBiOT...,8.3,top99,1941,tt0033467,https://www.imdb.com/title/tt0033467


In [8]:
for i in np.random.randint(0, 10, size=5):
    print(df.iloc[i]['title']," ",df.iloc[i]['description'])

Pulp Fiction   The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.
Schindler's List   In German-occupied Poland during World War II, industrialist Oskar Schindler gradually becomes concerned for his Jewish workforce after witnessing their persecution by the Nazis.
The Lord of the Rings: The Fellowship of the Ring   A meek Hobbit from the Shire and eight companions set out on a journey to destroy the powerful One Ring and save Middle-earth from the Dark Lord Sauron.
Pulp Fiction   The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.
The Lord of the Rings: The Fellowship of the Ring   A meek Hobbit from the Shire and eight companions set out on a journey to destroy the powerful One Ring and save Middle-earth from the Dark Lord Sauron.


## Data Transform

In [9]:
## Idea is to add the genre and date of release in the description
release_date = " The movie was released in the year "
genre = " The genre is of "
def add_date_genre(x):
    return x['description']+release_date+str(x['year'])+'.'+genre+' and '.join(eval(x['genre']))

In [10]:
df['description_2'] = df.apply(lambda x: add_date_genre(x),axis=1)

In [11]:
## Print few descriptions to check the append of genre and release date
for i in np.random.randint(0, 10, size=5):
    print(df.iloc[i]['title']," ",df.iloc[i]['description_2'])

The Shawshank Redemption   Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency. The movie was released in the year 1994. The genre is of Drama
The Good, the Bad and the Ugly   A bounty hunting scam joins two men in an uneasy alliance against a third in a race to find a fortune in gold buried in a remote cemetery. The movie was released in the year 1966. The genre is of Adventure and Western
The Shawshank Redemption   Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency. The movie was released in the year 1994. The genre is of Drama
The Lord of the Rings: The Fellowship of the Ring   A meek Hobbit from the Shire and eight companions set out on a journey to destroy the powerful One Ring and save Middle-earth from the Dark Lord Sauron. The movie was released in the year 2001. The genre is of Action and Adventure and Drama
The Good, the Bad and the Ugly   A bou

In [12]:
## Stemming and Lemmetizing the corpus and search query
def preprocess(text):
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]  # Apply stemming first
    lemmatized_words = [lemmatizer.lemmatize(word) for word in stemmed_words]  # Apply lemmatization
    return " ".join(lemmatized_words)

In [13]:
df['description_final'] = df.apply(lambda x: preprocess(x['description_2']),axis=1)
search_sentence_final = preprocess(search_sentence)

In [14]:
## Print few descriptions to check the stemm and lemmetization
for i in np.random.randint(0, 10, size=5):
    print(df.iloc[i]['title']," ",df.iloc[i]['description_final'])

12 Angry Men   the juri in a new york citi murder trial is frustrat by a singl member whose skeptic caution forc them to more care consid the evid befor jump to a hasti verdict. the movi wa releas in the year 1957. the genr is of crime and drama
The Shawshank Redemption   two imprison men bond over a number of years, find solac and eventu redempt through act of common decency. the movi wa releas in the year 1994. the genr is of drama
The Lord of the Rings: The Fellowship of the Ring   a meek hobbit from the shire and eight companion set out on a journey to destroy the power one ring and save middle-earth from the dark lord sauron. the movi wa releas in the year 2001. the genr is of action and adventur and drama
Schindler's List   in german-occupi poland dure world war ii, industrialist oskar schindler gradual becom concern for hi jewish workforc after wit their persecut by the nazis. the movi wa releas in the year 1993. the genr is of biographi and drama and histori
The Lord of the Rin

In [15]:
## Calculate TFIDF
tfidf_matrix = tfidf.fit_transform(df['description_final'].to_list()+[search_sentence_final])
word2tfidf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

In [16]:
## Got the tfidf score of each word in corpus
word2tfidf

{'000': np.float64(4.931825632724326),
 '10': np.float64(4.931825632724326),
 '12': np.float64(4.526360524616162),
 '17': np.float64(4.931825632724326),
 '1890': np.float64(4.931825632724326),
 '1920': np.float64(4.526360524616162),
 '1931': np.float64(4.526360524616162),
 '1936': np.float64(4.931825632724326),
 '1940': np.float64(4.931825632724326),
 '1941': np.float64(4.931825632724326),
 '1942': np.float64(4.931825632724326),
 '1946': np.float64(4.931825632724326),
 '1950': np.float64(4.931825632724326),
 '1952': np.float64(4.526360524616162),
 '1954': np.float64(4.526360524616162),
 '1957': np.float64(4.238678452164381),
 '1960': np.float64(4.931825632724326),
 '1962': np.float64(4.526360524616162),
 '1963': np.float64(4.526360524616162),
 '1964': np.float64(4.931825632724326),
 '1966': np.float64(4.931825632724326),
 '1968': np.float64(4.526360524616162),
 '1972': np.float64(4.931825632724326),
 '1974': np.float64(4.931825632724326),
 '1975': np.float64(4.931825632724326),
 '1977'

In [17]:
## Get Distil Bert Embeddings and combine tfidf with bert embeddings
def sentence_embedding(sentence, model, tokenizer, word2tfidf):
    tokens = tokenizer.tokenize(sentence)
    input_ids = tokenizer.encode(sentence, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model(input_ids)
    
    word_embeddings = outputs.last_hidden_state.squeeze(0)[1:-1]  # Ignore [CLS] and [SEP]
    embeddings = []
    
    for i, word in enumerate(tokens):
        if word in word2tfidf:  
            weight = word2tfidf.get(word, 1.0)
            embeddings.append(word_embeddings[i] * weight)
    
    if embeddings:
        return torch.mean(torch.stack(embeddings), dim=0).numpy()
    else:
        return np.zeros(model.config.hidden_size)

In [18]:
document_embeddings = np.array([sentence_embedding(sent, model, tokenizer, word2tfidf) for sent in df['description_final'].to_list()])

In [21]:
document_embeddings.shape

(100, 768)

: 

In [19]:
query_embeddings = np.array([sentence_embedding(search_sentence_final, model, tokenizer, word2tfidf)])

In [20]:
query_embeddings.shape

(1, 768)

## Inference

In [21]:
## function to calculate cosine distance, larger the value smaller the angle and hence greater the similarity
def similarity(x,y):
    cos = np.dot(x,y)
    res = cos/(np.linalg.norm(x)*np.linalg.norm(y))
    return res.item()

In [22]:
res = []
for emb in document_embeddings:
    res.append(similarity(query_embeddings,emb))

In [23]:
df['similarity'] = res

In [24]:
df.sample(5)

,rank,title,description,image,big_image,genre,thumbnail,rating,id,year,imdbid,imdb_link,description_2,description_final,similarity
0,1,The Shawshank Redemption,Two imprisoned men bond over a number of years...,https://m.media-amazon.com/images/M/MV5BMDFkYT...,https://m.media-amazon.com/images/M/MV5BMDFkYT...,['Drama'],https://m.media-amazon.com/images/M/MV5BMDFkYT...,9.3,top1,1994,tt0111161,https://www.imdb.com/title/tt0111161,Two imprisoned men bond over a number of years...,"two imprison men bond over a number of years, ...",0.765464
27,28,The Green Mile,The lives of guards on Death Row are affected ...,https://m.media-amazon.com/images/M/MV5BMTUxMz...,https://m.media-amazon.com/images/M/MV5BMTUxMz...,"['Crime', 'Drama', 'Fantasy']",https://m.media-amazon.com/images/M/MV5BMTUxMz...,8.6,top28,1999,tt0120689,https://www.imdb.com/title/tt0120689,The lives of guards on Death Row are affected ...,the live of guard on death row are affect by o...,0.764650
68,69,Inglourious Basterds,"In Nazi-occupied France during World War II, a...",https://m.media-amazon.com/images/M/MV5BOTJiND...,https://m.media-amazon.com/images/M/MV5BOTJiND...,"['Adventure', 'Drama', 'War']",https://m.media-amazon.com/images/M/MV5BOTJiND...,8.4,top69,2009,tt0361748,https://www.imdb.com/title/tt0361748,"In Nazi-occupied France during World War II, a...","in nazi-occupi franc dure world war ii, a plan...",0.756069
81,82,Come and See,"After finding an old rifle, a young boy joins ...",https://m.media-amazon.com/images/M/MV5BODM4Nj...,https://m.media-amazon.com/images/M/MV5BODM4Nj...,"['Drama', 'Thriller', 'War']",https://m.media-amazon.com/images/M/MV5BODM4Nj...,8.4,top82,1985,tt0091251,https://www.imdb.com/title/tt0091251,"After finding an old rifle, a young boy joins ...","after find an old rifle, a young boy join the ...",0.762214
55,56,Apocalypse Now,A U.S. Army officer serving in Vietnam is task...,https://m.media-amazon.com/images/M/MV5BYmQyNT...,https://m.media-amazon.com/images/M/MV5BYmQyNT...,"['Drama', 'Mystery', 'War']",https://m.media-amazon.com/images/M/MV5BYmQyNT...,8.4,top56,1979,tt0078788,https://www.imdb.com/title/tt0078788,A U.S. Army officer serving in Vietnam is task...,a u.s. armi offic serv in vietnam is task with...,0.763500


In [33]:
sim_arg_k = np.argsort(res)[-top_k_matches:][::-1]

In [34]:
sim_arg_k

array([94, 49, 78, 53, 77])

In [35]:
## Final Output
df.iloc[sim_arg_k][['title','similarity','description_2']]

,title,similarity,description_2
94,The Hunt,0.852198,"A teacher lives a lonely life, all the while s..."
49,Cinema Paradiso,0.822197,A filmmaker recalls his childhood when falling...
78,3 Idiots,0.821403,Two friends are searching for their long lost ...
53,City Lights,0.817958,"With the aid of a wealthy erratic tippler, a d..."
77,Your Name.,0.815180,Two strangers find themselves linked in a biza...
